# ODE discretization: Euler, trapezoidal, and RK4

**Learning goals:** compare numerical trajectories and estimate each method's convergence order.

**Predict first:** which methods remain accurate when the step size is halved, and by what factor should their terminal errors shrink?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

rate, horizon, x0 = -2.0, 3.0, 1.0

def integrate(method, step):
    times = np.arange(0.0, horizon + 0.5 * step, step)
    values = np.empty_like(times)
    values[0] = x0
    for k in range(len(times) - 1):
        x = values[k]
        if method == "Euler":
            values[k + 1] = x + step * rate * x
        elif method == "Trapezoidal":
            values[k + 1] = x * (1 + 0.5 * step * rate) / (1 - 0.5 * step * rate)
        elif method == "RK4":
            z = step * rate
            values[k + 1] = x * (1 + z + z**2 / 2 + z**3 / 6 + z**4 / 24)
    return times, values

reference = solve_ivp(lambda t, x: rate * x, (0, horizon), [x0], rtol=1e-12, atol=1e-14, dense_output=True)
for method in ["Euler", "Trapezoidal", "RK4"]:
    t, x = integrate(method, 0.25)
    plt.plot(t, x, "o-", label=method)
grid = np.linspace(0, horizon, 300)
plt.plot(grid, reference.sol(grid)[0], "k--", label="reference")
plt.legend()
plt.xlabel("Time")
plt.ylabel("State")
plt.show()

In [ ]:
steps = np.array([0.5, 0.25, 0.125, 0.0625])
errors = {}
exact_terminal = np.exp(rate * horizon)
for method in ["Euler", "Trapezoidal", "RK4"]:
    errors[method] = np.array([abs(integrate(method, h)[1][-1] - exact_terminal) for h in steps])
    order = np.polyfit(np.log(steps), np.log(errors[method]), 1)[0]
    print(f"{method:12s} estimated order: {order:.2f}")

**Experiment:** change `rate` to `-10`. Increase the step size until explicit Euler becomes unstable. Why does the trapezoidal update behave differently?

In [ ]:
assert errors["Euler"][-1] < errors["Euler"][0]
assert errors["Trapezoidal"][-1] < errors["Euler"][-1]
assert errors["RK4"][-1] < errors["Trapezoidal"][-1]
print("Checks passed.")